In [1]:
import sys
print(sys.version)

# Please switch to colab 2026/7 version to test

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
%pip install -q --no-cache-dir \
  "tokenizers==0.21.4" \
  "transformers==4.53.2" \
  "syncode==0.4.16" \
  accelerate

from syncode import Syncode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.1/206.1 kB 20.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 276.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 200.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 459.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 317.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 416.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1

In [5]:
EXTRACTION_GRAMMAR = r"""
start: "[" SIGNED_NUMBER "," SIGNED_NUMBER "]" OPERATOR
OPERATOR: "+" | "-" | "*" | "/"
%import common.SIGNED_NUMBER
%import common.WS
%ignore WS
"""

In [6]:
syn_llm = Syncode(
    model="microsoft/phi-2",
    grammar=EXTRACTION_GRAMMAR,
    mode="grammar_strict",
    parse_output_only=True,
    max_new_tokens=50,
    do_sample=False
)

[2026-10-27 20:17:47,315-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

[2026-10-27 20:19:09,785-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_8489793588_50257.pkl exist: False
[2026-10-27 20:19:09,786-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_8489793588_50257.pkl.
[2026-10-27 20:19:10,330-syncode.mask_store.fsm_set] - 6 FSMs with 37 states initialized in 0.01 seconds
[2026-10-27 20:19:10,331-syncode.mask_store.mask_store] - Ignore whitespace is True
[2026-10-27 20:19:10,336-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 36 to 11


100%|██████████| 37/37 [00:07<00:00,  4.95it/s]

[2026-10-27 20:19:17,852-syncode.mask_store.mask_store] - Time taken to create mask store: 8.04 seconds


In [7]:
FEW_SHOT_PREFIX = """
Extract the two operands in their original order and the arithmetic operator.
Use the format: [operand1, operand2] operator
Do not calculate the result.
Question: What is 327. multiplied by 11.0?
Extraction: [327.0, 11.0] *
Question: What is 45.1 plus 23.54?
Extraction: [45.1, 23.54] +
Question: What is 120.4 divided by 4.0?
Extraction: [120.4, 4.0] /
Question: What is 88.5 minus 12.25?
Extraction: [88.5, 12.25] -
""".strip()

def make_prompt(textual_query):
  return (
      FEW_SHOT_PREFIX
      + "\n\nQuestion: "
      + textual_query
      + "\nExtraction:"
  )

In [8]:
def parse_extraction(output):
    number_text, operator = output.strip()[1:].split("]")
    operands = [float(value) for value in number_text.split(",")]
    return operands, operator.strip()

def calculate_answer(operands, operator):
    a, b = operands
    if operator == "+":
        return a+b
    elif operator == "-":
        return a-b
    elif operator == "*":
        return a*b
    else:
        return a/b

In [9]:
def run_syncode(textual_query):
  prompt = make_prompt(textual_query)
  output = syn_llm.infer(prompt)[0]
  operands, operator = parse_extraction(output)
  final_answer = calculate_answer(operands, operator)
  return operands, operator, final_answer

In [10]:
test_queries = [
    "What is 327. multiplied by 11.0?",
    "What is 45.1 plus 23.54?",
    "What is 120.4 divided by 4.0?"
]
for query in test_queries:
  print(run_syncode(query))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)
([120.4, 4.0], '/', 30.1)
